# 02 · Index and retrieval

How each retrieval method finds the evidence for a question, then the full ablation on the **dev** split.

Dense and reranker results appear only after `python scripts/run_models.py` has cached the model outputs in `data/index/` (needs torch + Hugging Face; see `requirements-models.txt`). Everything below needs only numpy.

In [ ]:
import subprocess
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.rag.dataset import load_chunks, load_golden, load_gold_chunks, embed_text
from src.rag.retrieve import BM25, TOKENIZERS, top

chunks = load_chunks()
ids, texts = [c['id'] for c in chunks], [embed_text(c) for c in chunks]
loc = {c['id']: f"{c['doc_id']} · {c['loc']}" for c in chunks}
dev = load_golden('dev')
gold = load_gold_chunks()
len(chunks), len(dev)

## Syllables or words?

Vietnamese words are often two syllables (*học bổng*, *rèn luyện*). Word segmentation (pyvi) joins them, which makes terms more specific but can also split a query and a chunk differently.

In [ ]:
q = 'Học bổng khuyến khích cần GPA bao nhiêu?'
for name, tok in TOKENIZERS.items():
    print(name, tok(q))

In [ ]:
bm25 = {name: (BM25([tok(t) for t in texts]), tok) for name, tok in TOKENIZERS.items()}

def show(question, k=5):
    for name, (bm, tok) in bm25.items():
        print(f'--- BM25 {name}')
        for cid, s in top(ids, bm.scores(tok(question)), k):
            print(f'{s:6.2f}  {loc[cid]}')

show(q)

A question phrased differently from the text (*qua môn* vs *đạt*) is where lexical search struggles and dense search should help:

In [ ]:
show('Bao nhiêu điểm thì qua môn?')

## Full ablation (dev)

In [ ]:
out = subprocess.run([sys.executable, '-W', 'ignore', '../scripts/eval_retrieval.py'], capture_output=True, text=True)
from IPython.display import Markdown
Markdown(out.stdout)